# Sutram SRM — training on SEN2VENµS (free Colab)

Fine-tunes our ×4 super-resolution model on **real Sentinel-2 / VENµS pairs**,
including **KUDALIAR (Telangana, India)**.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Everything checkpoints to Google Drive every epoch, so a disconnect costs at most one
epoch — just re-run the notebook top to bottom and training resumes automatically.


## 1. Check GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Mount Drive

All state lives under `MyDrive/sutram_srm/` so nothing is lost when the VM dies.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/sutram_srm')
(DRIVE/'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE/'data').mkdir(parents=True, exist_ok=True)
print('drive ready:', DRIVE)


## 3. Get the code

Either clone your repo, or upload the `src/` and `scripts/` folders.
Set `REPO_URL` once you have pushed to GitHub.


In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/<you>/sutram-srm.git'

import os, pathlib
if REPO_URL:
    !git clone -q $REPO_URL /content/srm
else:
    # Fallback: expect the repo to have been copied to Drive.
    !cp -r /content/drive/MyDrive/sutram_srm/code /content/srm
os.chdir('/content/srm')
print(sorted(p.name for p in pathlib.Path('.').iterdir()))


## 4. Install dependencies

`sen2sr` supplies the SPAN backbone we fine-tune. Colab already has torch.


In [ ]:
!pip install -q sen2sr mlstac rasterio rioxarray pandas 2>&1 | tail -2
print('installed')


## 5. Download SEN2VENµS sites

`KUDALIAR` is the India site (Telangana, 7269 patches) and is the one that matters
for the jury. The others add biome diversity so the model does not overfit to one
landscape. Downloads land in Drive, so this only runs once.


In [ ]:
SITES = 'KUDALIAR'          # add ',ANJI,MAD-AMBO,SUDOUE-4' for more diversity
RAW = DRIVE/'data'/'sen2venus'

!python scripts/fetch_sen2venus.py --sites $SITES --out $RAW


## 6. Pack into training shards

**Scale note.** SEN2VENµS pairs are 10 m Sentinel-2 against 5 m VENµS — that is ×2.
Our deployment target is ×4 (10 m → 2.5 m), and no open 2.5 m reference exists.
So in `--scale 4` we degrade the *input* to 20 m with the Sentinel-2 PSF and keep the
**real** 5 m VENµS as the target: a genuine ×4 problem with a genuine high-resolution
reference. Only the input is synthetic, and it is synthesised with the sensor's own
response rather than bicubic.


In [ ]:
DATA = DRIVE/'data'/'sen2venus_x4'

!python scripts/build_dataset.py \
    --root $RAW --sites $SITES --scale 4 \
    --max-per-site 6000 --shard-size 1000 --out $DATA

import json
print(json.loads((DATA/'manifest.json').read_text())['sites'])


## 7. Train

Warm-starts from the released SEN2SRLite weights, so this converges in hours rather
than days. Re-running this cell after a disconnect resumes from `last.pt`.

Objective (see `src/srm/train/losses.py`): L1 + **LR-consistency through the sensor
PSF** + spectral angle + gradient. Notably *not* VGG-19 perceptual loss, which is
trained on 8-bit photographs and mismatched to 16-bit reflectance.


In [ ]:
CK = DRIVE/'checkpoints'
RESUME = CK/'last.pt' if (CK/'last.pt').exists() else ''

!python scripts/train.py \
    --data $DATA --out $CK \
    --epochs 40 --batch-size 16 --lr 2e-4 \
    --device cuda --amp --resume "$RESUME"


## 8. Training curves


In [ ]:
import json, matplotlib.pyplot as plt
h = json.loads((CK/'history.json').read_text())
ep = [r['epoch'] for r in h]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(ep, [r['loss'] for r in h], label='total')
ax[0].plot(ep, [r['l1'] for r in h], label='L1')
ax[0].plot(ep, [r['consistency'] for r in h], label='LR-consistency')
ax[0].set_title('training loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(ep, [r.get('val_psnr') for r in h], color='tab:green')
ax[1].set_title('val PSNR (dB)'); ax[1].set_xlabel('epoch')
ax[2].plot(ep, [r.get('val_sam_deg') for r in h], color='tab:red')
ax[2].set_title('val SAM (deg, lower better)'); ax[2].set_xlabel('epoch')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

best = max(h, key=lambda r: r.get('val_psnr', -1e9))
print(f"best epoch {best['epoch']}: PSNR {best['val_psnr']:.2f}  "
      f"SSIM {best['val_ssim']:.4f}  SAM {best['val_sam_deg']:.3f}")


## 9. Benchmark against the other branches

Wald protocol: degrade ×4, super-resolve back, score against the original.


In [ ]:
!cp $CK/best.pt checkpoints/best.pt 2>/dev/null || mkdir -p checkpoints && cp $CK/best.pt checkpoints/best.pt
!python scripts/make_test_scene.py
!python scripts/evaluate.py --input data/raw/test_ref_2p5m.tif \
    --branches bicubic,sen2sr,ours --device cuda


## 10. Visual check

Always look at the imagery. Metrics can improve while the output gets visibly worse —
that is precisely the failure mode this project exists to catch.


In [ ]:
import sys, warnings; sys.path.insert(0, 'src'); warnings.filterwarnings('ignore')
import numpy as np, matplotlib.pyplot as plt
from srm.train.dataset import Sen2VenusShards
from srm.models.ours_branch import OursBranch
from srm.models.bicubic_branch import BicubicBranch

ds = Sen2VenusShards(DATA, 'val', augment=False)
ours, bic = OursBranch(device='cuda'), BicubicBranch()

def show(a):
    rgb = a[:3].transpose(1, 2, 0)
    lo, hi = np.percentile(rgb, [2, 98])
    return np.clip((rgb - lo) / max(hi - lo, 1e-8), 0, 1)

fig, ax = plt.subplots(3, 4, figsize=(14, 10))
for r in range(3):
    lr, hr = ds[r * 7]
    lr, hr = lr.numpy(), hr.numpy()
    for c, (img, t) in enumerate([
        (lr, 'input 20 m'), (bic.predict(lr).sr, 'bicubic'),
        (ours.predict(lr).sr, 'ours'), (hr, 'VENuS 5 m truth')]):
        ax[r, c].imshow(show(img)); ax[r, c].set_title(t); ax[r, c].axis('off')
plt.tight_layout(); plt.show()


## 11. Keep the checkpoint

`best.pt` in Drive is the artefact to bring home. Copy it into `checkpoints/` in the
repo and the `Ours` branch is live in the local pipeline and the Streamlit demo.


In [ ]:
import shutil
print('best.pt  %.1f MB' % ((CK/'best.pt').stat().st_size/1e6))
print('download it, or keep it in Drive at:', CK/'best.pt')
